In [2]:
import numpy as np
import pandas as pd
from pathlib import Path
import matplotlib.pyplot as plt
import plotly.graph_objects as go

# ============================================================
# Directional Connectedness (TO / FROM) + Static Save + Interactive Plot
# ------------------------------------------------------------
# 입력
#   - gfevd_all.npy : shape (T_eff, N, N)
#     row = response variable
#     col = shock source
#
# 출력
#   - ./directional_output/directional_time.csv
#   - ./directional_output/directional_mean.csv
#   - ./directional_output/to_series.png
#   - ./directional_output/from_series.png
#
# 화면 출력
#   - Plotly interactive TO graph
#   - Plotly interactive FROM graph
# ============================================================

# =========================
# Config
# =========================
BASE_DIR = Path("./")
OUT_DIR = BASE_DIR / "directional_output"
OUT_DIR.mkdir(parents=True, exist_ok=True)

GFEVD_FILE = BASE_DIR / "gfevd_all.npy"

DATE_FILE_CANDIDATES = [
    BASE_DIR / "tvpvar_input_scaled.csv",
    BASE_DIR / "tvpvar_input_transformed.csv",
    BASE_DIR / "tvpvar_raw_level_merged.csv",
]

OUT_TIME = OUT_DIR / "directional_time.csv"
OUT_MEAN = OUT_DIR / "directional_mean.csv"
OUT_TO_PNG = OUT_DIR / "to_series.png"
OUT_FROM_PNG = OUT_DIR / "from_series.png"
OUT_TO_HTML = OUT_DIR / "to_series_interactive.html"
OUT_FROM_HTML = OUT_DIR / "from_series_interactive.html"

VAR_NAMES = [
    "SOLVPN",
    "COPPER",
    "GOLD",
    "SILVER",
    "DXY",
    "UST10Y",
    "VIX"
]

ROW_SUM_TOL = 1e-6
FORCE_ROW_NORMALIZE = True

# =========================
# Helper
# =========================
def load_dates(target_len: int):
    for fp in DATE_FILE_CANDIDATES:
        if fp.exists():
            try:
                df = pd.read_csv(fp)
                if "Date" not in df.columns:
                    continue

                dt = pd.to_datetime(df["Date"], errors="coerce").dropna().reset_index(drop=True)

                if len(dt) < target_len:
                    continue

                dt = dt.iloc[-target_len:].reset_index(drop=True)
                return dt

            except Exception as e:
                print(f"[WARN] Failed to read dates from {fp}: {e}")

    return None


def row_normalize_theta(theta: np.ndarray) -> np.ndarray:
    row_sum = theta.sum(axis=1, keepdims=True)
    row_sum[row_sum == 0] = 1.0
    return theta / row_sum


def compute_to_from(theta: np.ndarray):
    """
    theta: (N, N)
    row = response variable
    col = shock source
    """
    diag = np.diag(theta)

    # i가 다른 변수들로부터 받는 영향
    from_ = theta.sum(axis=1) - diag

    # i가 다른 변수들에게 주는 영향
    to_ = theta.sum(axis=0) - diag

    return to_, from_


# =========================
# Load
# =========================
if not GFEVD_FILE.exists():
    raise FileNotFoundError(f"Not found: {GFEVD_FILE}")

gfevd_all = np.load(GFEVD_FILE)

if gfevd_all.ndim != 3:
    raise ValueError(f"gfevd_all must be 3D, got shape={gfevd_all.shape}")

T_eff, N, N2 = gfevd_all.shape

if N != N2:
    raise ValueError(f"Last two dims must be square, got shape={gfevd_all.shape}")

if len(VAR_NAMES) != N:
    raise ValueError(f"len(VAR_NAMES)={len(VAR_NAMES)} must match N={N}")

print(f"[INFO] Loaded GFEVD: shape={gfevd_all.shape}")

# =========================
# Check / Normalize
# =========================
gfevd_proc = gfevd_all.copy()

row_sum_check = gfevd_proc.sum(axis=2)
max_row_err = np.max(np.abs(row_sum_check - 1.0))
print(f"[INFO] Max row-sum error before normalization: {max_row_err:.8f}")

if FORCE_ROW_NORMALIZE and max_row_err > ROW_SUM_TOL:
    print("[INFO] Applying row normalization to all theta_t")
    for t in range(T_eff):
        gfevd_proc[t] = row_normalize_theta(gfevd_proc[t])

row_sum_check_after = gfevd_proc.sum(axis=2)
max_row_err_after = np.max(np.abs(row_sum_check_after - 1.0))
print(f"[INFO] Max row-sum error after normalization: {max_row_err_after:.8f}")

# =========================
# Dates
# =========================
dates = load_dates(T_eff)

# =========================
# Time-varying TO / FROM
# =========================
records = []

for t in range(T_eff):
    theta = gfevd_proc[t]
    to_, from_ = compute_to_from(theta)

    row = {}
    if dates is not None:
        row["Date"] = dates.iloc[t]
    else:
        row["t"] = t

    for i, name in enumerate(VAR_NAMES):
        row[f"TO_{name}"] = to_[i] * 100.0
        row[f"FROM_{name}"] = from_[i] * 100.0

    records.append(row)

df_time = pd.DataFrame(records)

# =========================
# Mean TO / FROM
# =========================
rows_mean = []

for name in VAR_NAMES:
    rows_mean.append({
        "Variable": name,
        "TO_mean": df_time[f"TO_{name}"].mean(),
        "FROM_mean": df_time[f"FROM_{name}"].mean(),
    })

df_mean = pd.DataFrame(rows_mean)

# =========================
# Save CSV
# =========================
df_time.to_csv(OUT_TIME, index=False)
df_mean.to_csv(OUT_MEAN, index=False)

print("[INFO] Saved:")
print(f"  - {OUT_TIME}")
print(f"  - {OUT_MEAN}")

print("\n[INFO] Mean Directional Connectedness")
print(df_mean.to_string(index=False))

# =========================
# X-axis 준비
# =========================
if "Date" in df_time.columns:
    df_time["Date"] = pd.to_datetime(df_time["Date"])
    x = df_time["Date"]
    x_label = "Date"
else:
    x = df_time["t"]
    x_label = "t"

# =========================
# Matplotlib Save: TO
# =========================
plt.figure(figsize=(12, 6))
for name in VAR_NAMES:
    plt.plot(x, df_time[f"TO_{name}"], label=name, linewidth=1.2)

plt.title("Directional TO (Outgoing Spillover)")
plt.xlabel(x_label)
plt.ylabel("Contribution (%)")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(OUT_TO_PNG, dpi=300, bbox_inches="tight")
plt.close()

# =========================
# Matplotlib Save: FROM
# =========================
plt.figure(figsize=(12, 6))
for name in VAR_NAMES:
    plt.plot(x, df_time[f"FROM_{name}"], label=name, linewidth=1.2)

plt.title("Directional FROM (Incoming Spillover)")
plt.xlabel(x_label)
plt.ylabel("Contribution (%)")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(OUT_FROM_PNG, dpi=300, bbox_inches="tight")
plt.close()

print(f"  - {OUT_TO_PNG}")
print(f"  - {OUT_FROM_PNG}")

# =========================
# Plotly Interactive: TO
# =========================
fig_to = go.Figure()

for name in VAR_NAMES:
    fig_to.add_trace(
        go.Scatter(
            x=x,
            y=df_time[f"TO_{name}"],
            mode="lines",
            name=name
        )
    )

fig_to.update_layout(
    title="Interactive Directional TO (Outgoing Spillover)",
    xaxis_title=x_label,
    yaxis_title="Contribution (%)",
    height=600,
    hovermode="x unified"
)

fig_to.show()
fig_to.write_html(str(OUT_TO_HTML))

# =========================
# Plotly Interactive: FROM
# =========================
fig_from = go.Figure()

for name in VAR_NAMES:
    fig_from.add_trace(
        go.Scatter(
            x=x,
            y=df_time[f"FROM_{name}"],
            mode="lines",
            name=name
        )
    )

fig_from.update_layout(
    title="Interactive Directional FROM (Incoming Spillover)",
    xaxis_title=x_label,
    yaxis_title="Contribution (%)",
    height=600,
    hovermode="x unified"
)

fig_from.show()
fig_from.write_html(str(OUT_FROM_HTML))

print(f"  - {OUT_TO_HTML}")
print(f"  - {OUT_FROM_HTML}")

[INFO] Loaded GFEVD: shape=(1035, 7, 7)
[INFO] Max row-sum error before normalization: 0.00000000
[INFO] Max row-sum error after normalization: 0.00000000
[INFO] Saved:
  - directional_output\directional_time.csv
  - directional_output\directional_mean.csv

[INFO] Mean Directional Connectedness
Variable   TO_mean  FROM_mean
  SOLVPN 58.919029  54.290093
  COPPER 49.856591  51.335666
    GOLD 65.009679  58.934270
  SILVER 64.357477  57.881032
     DXY 49.836966  53.793084
  UST10Y 35.690339  45.984719
     VIX 44.741096  46.192313
  - directional_output\to_series.png
  - directional_output\from_series.png


  - directional_output\to_series_interactive.html
  - directional_output\from_series_interactive.html
